In [21]:
# Aggregate raw PV power CSV files into a single time-resolved dataset.

from pathlib import Path
import pandas as pd


COL_MAP = {
    "시간": "time",
    "표면온도(°C)": "surface_temperature",
    "대기온도(°C)": "ambient_temperature",
    "수평일사량(Wh/㎡)": "global_horizontal_irradiance",
    "입사일사량(Wh/㎡)": "plane_of_array_irradiance",
    "전압(V)": "voltage",
    "전류(A)": "current",
    "전력(kW)": "output_power",
    "발전량(kWh)": "energy_kwh",
}

# Analysis period covered in this study.
START_DATE = pd.Timestamp("2024-01-01")
END_DATE = pd.Timestamp("2024-12-31")


def build_timestamp(df: pd.DataFrame, file_date: str) -> pd.Series:
    """Construct a full timestamp from a file-level date and time-of-day strings."""
    time_str = df["time"].astype(str).str.strip()
    return pd.to_datetime(file_date + " " + time_str, errors="coerce")


def load_and_merge_power_csv(raw_dir: Path, encoding: str = "cp949") -> pd.DataFrame:
    """Load daily PV power CSV files, restrict to the target period, and concatenate them."""
    csv_paths = sorted(raw_dir.glob("_20*.csv"))
    if not csv_paths:
        raise FileNotFoundError(f"No CSV files found in: {raw_dir}")

    frames: list[pd.DataFrame] = []
    for p in csv_paths:
        file_date_str = p.stem.lstrip("_")
        file_date = pd.to_datetime(file_date_str, format="%Y-%m-%d", errors="coerce")

        if pd.isna(file_date) or not (START_DATE <= file_date <= END_DATE):
            continue

        df = pd.read_csv(p, encoding=encoding).rename(columns=COL_MAP)
        df["timestamp"] = build_timestamp(df, file_date_str)

        frames.append(df)

    if not frames:
        raise RuntimeError("No CSV files fell within the specified analysis period.")

    return pd.concat(frames, ignore_index=True)


def add_time_components(df: pd.DataFrame) -> pd.DataFrame:
    """Decompose the timestamp into explicit calendar and clock components."""
    df["month"] = df["timestamp"].dt.month
    df["day"] = df["timestamp"].dt.day
    df["hour"] = df["timestamp"].dt.hour
    df["minute"] = df["timestamp"].dt.minute
    return df


def main() -> None:
    work_dir = Path.cwd()
    raw_dir = work_dir / "Power_raw"

    df = load_and_merge_power_csv(raw_dir=raw_dir, encoding="cp949")
    df = add_time_components(df)

    out_path = work_dir / "hyanglin_power_20240101_20241231_merged.csv"
    df.to_csv(out_path, index=False, encoding="utf-8-sig")


if __name__ == "__main__":
    main()

In [22]:
# Integrate PV power data with KMA AWS observations using exact minute-level matching.

from __future__ import annotations

from pathlib import Path
import pandas as pd


def load_pv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig", parse_dates=["timestamp"])
    df["time_key"] = df["timestamp"].dt.floor("min")
    return df


def load_kma(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path, encoding="utf-8-sig")
    else:
        df = pd.read_excel(path)

    if "DATETIME" not in df.columns:
        if "YYMMDDHHMI" not in df.columns:
            raise ValueError("KMA file must contain 'DATETIME' or 'YYMMDDHHMI'.")
        df["DATETIME"] = pd.to_datetime(
            df["YYMMDDHHMI"].astype(str),
            format="%Y%m%d%H%M",
            errors="coerce",
        )

    df["kma_datetime"] = pd.to_datetime(df["DATETIME"], errors="coerce").dt.floor("min")
    df = df.dropna(subset=["kma_datetime"])
    df = df.drop_duplicates(subset=["kma_datetime"], keep="first")
    return df


def attach_kma_exact(pv_df: pd.DataFrame, kma_df: pd.DataFrame) -> pd.DataFrame:
    merged = pv_df.merge(
        kma_df.drop(columns=["DATETIME"], errors="ignore"),
        left_on="time_key",
        right_on="kma_datetime",
        how="inner",  # exact matches only
        suffixes=("", "_KMA"),
    )

    merged = merged.drop(columns=["time_key"])
    return merged


def main() -> None:
    work_dir = Path.cwd()
    project_root = work_dir.parent

    pv_path = work_dir / "hyanglin_power_20240101_20241231_merged.csv"
    kma_path = project_root / "1. KMA" / "aws_min_108_20240101_20241231_merged.xlsx"

    pv_df = load_pv(pv_path)
    kma_df = load_kma(kma_path)

    merged = attach_kma_exact(pv_df, kma_df)

    out_path = work_dir / "hyanglin_power_20240101_20241231_merged_with_kma_exact.csv"
    merged.to_csv(out_path, index=False, encoding="utf-8-sig")

    keep_rate = len(merged) / len(pv_df) * 100
    print(f"[INFO] saved: {out_path}")
    print(f"[INFO] kept PV rows (exact match): {keep_rate:.2f}% ({len(merged)}/{len(pv_df)})")


if __name__ == "__main__":
    main()

[INFO] saved: /home/serc/Seungtae/260207_hyanglin/2. PV_Power_Data/hyanglin_power_20240101_20241231_merged_with_kma_exact.csv
[INFO] kept PV rows (exact match): 100.00% (35136/35136)


In [23]:
# Remove variables not used for model training and construct a training-ready dataset.

from pathlib import Path
import pandas as pd

# Paths: input = PV+KMA merged CSV from previous cell; output = training-ready Excel.
BASE_DIR = Path.cwd()
INPUT_PATH = BASE_DIR / "hyanglin_power_20240101_20241231_merged_with_kma_exact.csv"
OUTPUT_PATH = BASE_DIR / "Hyanglin_dataset_merged.xlsx"

# Columns excluded from model training
DROP_COLUMNS = [
    "time",
    "voltage",
    "current",
    "energy_kwh",
    "timestamp",
    "YYMMDDHHMI",
    "STN",
    "kma_datetime",
]

# Load dataset (ensure the PV+KMA merged CSV from the previous cell exists)
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_PATH}. Run the previous cell (PV+KMA merge) first."
    )
df = pd.read_csv(INPUT_PATH)

# Identify daily keys (used to drop whole days with NaN or error codes).
if "timestamp" in df.columns:
    date_key = pd.to_datetime(df["timestamp"]).dt.date
else:
    # Fallback: derive a date key from calendar components when timestamp is absent.
    required_cols = {"month", "day"}
    if not required_cols.issubset(df.columns):
        raise ValueError("Cannot construct daily keys: missing 'timestamp' or 'month'/'day' columns.")
    # Year information is not available here; month-day pairs are used as daily identifiers.
    date_key = list(zip(df["month"], df["day"]))

# Days with any missing value in any column.
row_has_nan = df.isna().any(axis=1)
nan_days = sorted(set(date_key[row_has_nan]))
if nan_days:
    print("[INFO] dropping all rows for days with missing values (NaN):")
    for d in nan_days:
        print("  ", d)
    print(f"[INFO] number of days dropped due to NaN: {len(nan_days)}")
else:
    print("[INFO] no days with missing values (NaN) detected.")

# Days with any feature value <= -98 (error/missing code).
num_cols = df.select_dtypes(include=["number"]).columns
err_mask = (df[num_cols] <= -98).any(axis=1)
err_days = sorted(set(date_key[err_mask]))
if err_days:
    print("[INFO] dropping all rows for days with feature value <= -98 (error code):")
    for d in err_days:
        print("  ", d)
    print(f"[INFO] number of days dropped due to error code (<= -98): {len(err_days)}")
else:
    print("[INFO] no days with feature value <= -98 (error code) detected.")

# Drop all identified days (union of NaN days and error-code days).
if nan_days or err_days:
    days_to_drop = set(nan_days) | set(err_days)
    mask_keep = ~pd.Series(date_key).isin(days_to_drop)
    df = df.loc[mask_keep].reset_index(drop=True)

# Remove non-informative or redundant variables.
df = df.drop(columns=[c for c in DROP_COLUMNS if c in df.columns])

# Rename KMA AWS variables to descriptive feature names.
KMA_RENAME = {
    "WD1": "wind_direction_1min_avg",
    "WS1": "wind_speed_1min_avg",
    "WDS": "wind_direction_gust",
    "WSS": "wind_speed_gust",
    "WD10": "wind_direction_10min_avg",
    "WS10": "wind_speed_10min_avg",
    "TA": "air_temperature_1min_avg",
    "RE": "precipitation_detected",
    "RN-15m": "rainfall_15min_accumulated",
    "RN-60m": "rainfall_60min_accumulated",
    "RN-12H": "rainfall_12h_accumulated",
    "RN-DAY": "daily_rainfall_accumulated",
    "HM": "relative_humidity",
    "PA": "atmospheric_pressure_local",
    "PS": "atmospheric_pressure_sea_level",
    "TD": "dew_point_temperature",
}

rename_map = {k: v for k, v in KMA_RENAME.items() if k in df.columns}
if rename_map:
    df = df.rename(columns=rename_map)

# Enforce chronological ordering.
df = df.sort_values([
    "month",
    "day",
    "hour",
    "minute",
]).reset_index(drop=True)

# Move target column (output_power) to the last position
if "output_power" in df.columns:
    cols = [c for c in df.columns if c != "output_power"] + ["output_power"]
    df = df[cols]

# Export training-ready dataset
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_excel(OUTPUT_PATH, index=False)

[INFO] dropping all rows for days with missing values (NaN):
   2024-07-07
   2024-07-08
   2024-07-09
[INFO] number of days dropped due to NaN: 3
[INFO] dropping all rows for days with feature value <= -98 (error code):
   2024-02-06
   2024-02-10
   2024-02-22
   2024-03-11
   2024-03-12
   2024-03-14
   2024-03-18
   2024-07-02
   2024-07-15
   2024-07-19
   2024-07-31
   2024-08-01
   2024-09-07
   2024-11-15
   2024-11-16
   2024-12-03
   2024-12-05
   2024-12-06
[INFO] number of days dropped due to error code (<= -98): 18


In [24]:
# Diagnose the earliest and latest daily power generation times across the analysis period.

from pathlib import Path
import pandas as pd


def extract_extreme_daily_power_times(
    csv_path: str,
    time_col: str = "timestamp",
    power_col: str = "output_power",
    power_threshold: float = 0.0,
    encoding: str | None = None,
):
    """
    Find:
      (1) The earliest 'first power time-of-day' across all dates.
      (2) The latest  'last  power time-of-day'  across all dates.
    Returns both time-of-day and the corresponding dates.
    """

    df = pd.read_csv(csv_path, encoding=encoding)

    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df = df.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)

    df_day = df[df[power_col] > power_threshold].copy()
    if df_day.empty:
        raise ValueError(f"No samples found with {power_col} > {power_threshold}")

    df_day["date"] = df_day[time_col].dt.date

    daily_bounds = (
        df_day.groupby("date")[time_col]
        .agg(first_power_time="min", last_power_time="max")
        .reset_index()
    )

    daily_bounds["first_tod"] = pd.to_datetime(daily_bounds["first_power_time"]).dt.time
    daily_bounds["last_tod"] = pd.to_datetime(daily_bounds["last_power_time"]).dt.time

    # Earliest first time-of-day across all days
    idx_earliest_first = daily_bounds["first_tod"].astype(str).idxmin()
    earliest_first_row = daily_bounds.loc[idx_earliest_first]

    # Latest last time-of-day across all days
    idx_latest_last = daily_bounds["last_tod"].astype(str).idxmax()
    latest_last_row = daily_bounds.loc[idx_latest_last]

    return {
        "earliest_first": {
            "date": earliest_first_row["date"],
            "time_of_day": earliest_first_row["first_tod"],
            "timestamp": earliest_first_row["first_power_time"],
        },
        "latest_last": {
            "date": latest_last_row["date"],
            "time_of_day": latest_last_row["last_tod"],
            "timestamp": latest_last_row["last_power_time"],
        },
        "daily_bounds": daily_bounds,
    }


if __name__ == "__main__":
    base_dir = Path.cwd()
    csv_path = base_dir / "hyanglin_power_20240101_20241231_merged_with_kma_exact.csv"

    res = extract_extreme_daily_power_times(
        csv_path=str(csv_path),
        time_col="timestamp",
        power_col="output_power",
        power_threshold=0.0,
        encoding=None,  # Use "cp949" if required.
    )

    print("[Earliest first power time-of-day across all dates]")
    print(f"date       : {res['earliest_first']['date']}")
    print(f"time_of_day: {res['earliest_first']['time_of_day']}")
    print(f"timestamp  : {res['earliest_first']['timestamp']}")

    print("\n[Latest last power time-of-day across all dates]")
    print(f"date       : {res['latest_last']['date']}")
    print(f"time_of_day: {res['latest_last']['time_of_day']}")
    print(f"timestamp  : {res['latest_last']['timestamp']}")

[Earliest first power time-of-day across all dates]
date       : 2024-05-22
time_of_day: 05:15:00
timestamp  : 2024-05-22 05:15:00

[Latest last power time-of-day across all dates]
date       : 2024-05-27
time_of_day: 19:45:00
timestamp  : 2024-05-27 19:45:00


In [25]:
import os
from pathlib import Path
import pandas as pd


def filter_daytime_and_save_new_file(
    input_excel: str,
    start_hour: int = 5,
    end_hour: int = 19,
    hour_col: str = "hour",
    suffix: str = "_daytime_5_19",
):
    """
    Filter rows by hour range and save to a new Excel file
    without modifying the original file.
    """

    # Load data
    df = pd.read_excel(input_excel)

    if hour_col not in df.columns:
        raise KeyError(f"'{hour_col}' not found in columns.")

    # Filter by hour range
    df_filtered = df[(df[hour_col] >= start_hour) & (df[hour_col] <= end_hour)].copy()

    # Generate new file path
    base, ext = os.path.splitext(input_excel)
    output_excel = f"{base}{suffix}{ext}"

    # Save as new file
    df_filtered.to_excel(output_excel, index=False)

    print(f"Original file preserved : {input_excel}")
    print(f"Filtered file saved     : {output_excel}")
    print(f"Rows before filtering  : {len(df)}")
    print(f"Rows after filtering   : {len(df_filtered)}")

    return output_excel


if __name__ == "__main__":
    base_dir = Path.cwd()
    input_excel = base_dir / "Hyanglin_dataset_merged.xlsx"

    filter_daytime_and_save_new_file(
        input_excel=str(input_excel),
        start_hour=5,
        end_hour=19,
        hour_col="hour",
        suffix="_daytime_5_19",
    )

Original file preserved : /home/serc/Seungtae/260207_hyanglin/2. PV_Power_Data/Hyanglin_dataset_merged.xlsx
Filtered file saved     : /home/serc/Seungtae/260207_hyanglin/2. PV_Power_Data/Hyanglin_dataset_merged_daytime_5_19.xlsx
Rows before filtering  : 33120
Rows after filtering   : 20700
